# B2-024-gpu-scientific-ml-capstone — Practice p02 — Solution

**Type:** mc · **Difficulty:** intro · **Concepts:** gpu-colab-l4-workflow

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Solution

**Reading the traceback in its three parts (Session 1, §2).**

1. *Devices:* `cuda:0 and cpu`. One operand of some operation was moved to the GPU and one was not.
2. *Operation:* `wrapper_CUDA__cudnn_convolution`. The failing call is a convolution, i.e. the first layer `conv1` of `ShapeCNN`, which is the very first operation of the forward pass.
3. *Argument:* `input`. The convolution's *input* tensor (the image batch) is on the wrong device; the weight side (which would be reported as `weight`) is fine.

So the CUDA convolution received GPU weights and a CPU image batch.
The cause is in the loop: `images.to(device)` and `labels.to(device)` are tensor `.to` calls, which **return a new tensor** and never modify the original.
The results are discarded, so `images` is still the CPU batch when `model(images)` runs.
(`model = ShapeCNN().to(device)` is fine: module `.to` moves parameters in place and returns the same module, and the result is even assigned.)

**Fix:** `images = images.to(device)` and `labels = labels.to(device)`.

**Why the CPU run raised nothing.** On the laptop `torch.cuda.is_available()` is false, so `device` is `cpu`; the model's parameters and the batch are both already on the CPU, `images.to(device)` is a no-op returning the same tensor, and no mixing of devices ever happens. The bug is latent until a second device exists.

**Answer: B.**

**Rejected options.**

* **A** is wrong because `nn.Module.to` moves registered parameters *in place* and returns the same module, so the model *is* on the GPU (and the traceback names the `input` argument, not the weight).
* **C** is wrong because `cpu` is the host processor, not a second accelerator card; `cuda:0` versus `cpu` means "GPU tensor versus host tensor", and restricting GPUs changes nothing.
* **D** is wrong because autocast changes dtypes of selected operations, never devices; removing it would leave the CPU batch in place and the same error.
* **E** is wrong because `module.to` keeps the same parameter objects, so an optimizer created after the move references the GPU parameters; and the error is raised in the forward convolution before the optimizer ever steps.

The cell below reproduces the failure and the fix on any machine, using the data-free `meta` device as the "second device" (Session 1, §2): the model lives on `meta`, the batch on the CPU.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(20261022)
dev = torch.device("meta")                       # stand-in for cuda:0 on a CPU-only machine
model = nn.Sequential(nn.Conv2d(1, 8, 3), nn.ReLU(), nn.Flatten(), nn.Linear(8 * 6 * 6, 3))
model = model.to(dev)
images = torch.rand(4, 1, 8, 8)
labels = torch.tensor([0, 1, 2, 0])

# Buggy pattern: results of tensor .to are discarded
images.to(dev)
labels.to(dev)
bug_raised = False
try:
    model(images)
except (RuntimeError, NotImplementedError) as err:
    bug_raised = True
    print("buggy loop raises:", type(err).__name__, str(err).splitlines()[0][:120])
print("images still on:", images.device, "| model weight on:", next(model.parameters()).device)

# Fixed pattern: keep the returned tensors
images = images.to(dev)
labels = labels.to(dev)
logits = model(images)
print("fixed loop logits on:", logits.device, tuple(logits.shape))

# Module .to returns the same object (rules out A)
m = nn.Linear(2, 2)
same_object = m.to("cpu") is m
ANSWER = "B"
print("module .to returns same object:", same_object)

### Answer check

In [ ]:
assert bug_raised
assert images.device.type == "meta" and logits.device.type == "meta" and logits.shape == (4, 3)
assert same_object
assert ANSWER == "B"